# services

> The service model: a plain dict per service, validated, rendered to Caddy blocks and guarded file-write cmds

In [ ]:
#| default_exp services

In [ ]:
#| hide
from nbdev.showdoc import *

A service is `{"name", "domain", "port", "public", "packages", "cmds"}`.
`domain=None` → no Caddy block (outbound-only services like discopipe).
`public=False` → `@vpn` gate; `public=True` → plain TLS reverse_proxy.
Fragments express file writes (systemd units, CLAUDE.md, ...) as runcmd
lines via `write_file_cmd`, the same heredoc idiom as `caddy_cmds()`.

## `check_service`

In [ ]:
#| export
def check_service(
    svc:dict,  # service dict: name, domain, port, public, packages, cmds
)->dict:       # the same dict, validated
    """Validate a service dict against the service model; return it unchanged."""
    if not isinstance(svc, dict):
        raise ValueError(f"service must be a dict, got {type(svc).__name__}")
    name = svc.get("name")
    if not isinstance(name, str) or not name.strip():
        raise ValueError(f"service name must be a non-empty string, got {name!r}")
    domain, port = svc.get("domain"), svc.get("port")
    public = svc.get("public", False)
    if not isinstance(public, bool):
        raise ValueError(f"{name}: public must be a bool, got {public!r}")
    if domain is None:
        if public:
            raise ValueError(f"{name}: domain=None with public=True (nothing to publish)")
    else:
        if isinstance(port, bool) or not isinstance(port, int) or not 1 <= port <= 65535:
            raise ValueError(f"{name}: a service with a domain needs a port in 1-65535, got {port!r}")
    for key in ("packages", "cmds"):
        val = svc.get(key, [])
        if not isinstance(val, list) or not all(isinstance(x, str) for x in val):
            raise ValueError(f"{name}: {key} must be a list of str, got {val!r}")
    return svc

In [ ]:
ok = {"name": "reconcile", "domain": "reconcile.ninjalabo.ai",
      "port": 5001, "public": False, "packages": [], "cmds": []}
assert check_service(ok) is ok

nodomain = {"name": "discopipe", "domain": None, "port": None, "public": False,
            "packages": ["python3-venv"], "cmds": ["true"]}
assert check_service(nodomain) is nodomain

for bad in (
    {},                                                        # no name
    {"name": ""},                                              # empty name
    {"name": "x", "domain": None, "public": True},             # nothing to publish
    {"name": "x", "domain": "a.b.c", "port": None},            # domain needs a port
    {"name": "x", "domain": "a.b.c", "port": 70000},           # port out of range
    {"name": "x", "domain": "a.b.c", "port": 80, "public": 1}, # public not a bool
    {"name": "x", "domain": None, "packages": "notalist"},     # packages not a list
    {"name": "x", "domain": None, "cmds": [1]},                # cmds items must be str
    "not-a-dict",
):
    try:
        check_service(bad); assert False, f"must raise: {bad!r}"
    except ValueError: pass

## `service_site`

In [ ]:
#| export
from hetznerinit.caddy import caddy_site

def service_site(
    svc:dict,     # a service dict (validated here via check_service)
)->str|None:      # Caddyfile site block, or None when the service has no domain
    """Render a service's Caddy site block; VPN-gated unless public."""
    svc = check_service(svc)
    if svc.get("domain") is None:
        return None
    if svc.get("public", False):
        return caddy_site(svc["domain"], svc["port"], vpn_subnet=None)
    return caddy_site(svc["domain"], svc["port"])

In [ ]:
assert service_site({"name": "d", "domain": None, "public": False}) is None

gated = service_site({"name": "r", "domain": "reconcile.ninjalabo.ai", "port": 5001, "public": False})
assert "@vpn remote_ip 10.0.0.0/24" in gated
assert "reverse_proxy 127.0.0.1:5001" in gated and "respond 403" in gated

pub = service_site({"name": "r", "domain": "reconcile.ninjalabo.ai", "port": 5001, "public": True})
assert "@vpn" not in pub and "respond 403" not in pub
assert "reverse_proxy 127.0.0.1:5001" in pub

## `write_file_cmd`

In [ ]:
#| export
import re

def write_file_cmd(
    path:str,                 # absolute path to write on the box
    content:str,              # file content (must not contain the heredoc delimiter)
    owner:str="root:root",    # chown owner:group
    mode:str="0644",          # chmod mode
)->str:                       # one cloud-init runcmd line (heredoc write + chown + chmod)
    """Return a guarded-heredoc runcmd line that writes content to path, then sets owner and mode."""
    # every value below lands in a root shell: whitelist instead of quoting
    # (same rule as hetznerinit's wg_server_cmds)
    if not isinstance(path, str) or not re.fullmatch(r"/[A-Za-z0-9._/-]+", path):
        raise ValueError(f"path must be absolute using [A-Za-z0-9._/-], got {path!r}")
    if not isinstance(owner, str) or not re.fullmatch(r"[a-z_][a-z0-9_-]*:[a-z_][a-z0-9_-]*", owner):
        raise ValueError(f"owner must be user:group, got {owner!r}")
    if not isinstance(mode, str) or not re.fullmatch(r"0?[0-7]{3}", mode):
        raise ValueError(f"mode must be 3 octal digits (optional 0 prefix), got {mode!r}")
    if "WF_EOF" in content:
        raise ValueError("content must not contain the heredoc delimiter 'WF_EOF'")
    return (f"cat > {path} <<'WF_EOF'\n{content.rstrip(chr(10))}\nWF_EOF\n"
            f"chown {owner} {path}\nchmod {mode} {path}")

In [ ]:
cmd = write_file_cmd("/etc/foo.conf", "line1\nline2\n", owner="discopipe:discopipe", mode="0600")
assert cmd.startswith("cat > /etc/foo.conf <<'WF_EOF'\n")
assert "line1\nline2\nWF_EOF" in cmd
assert "chown discopipe:discopipe /etc/foo.conf" in cmd
assert "chmod 0600 /etc/foo.conf" in cmd

# same guard as caddy_cmds: delimiter in the payload must raise
try:
    write_file_cmd("/etc/foo", "evil\nWF_EOF\nrm -rf /"); assert False, "guard must raise"
except ValueError: pass

# root-shell inputs are whitelisted, not quoted (same rule as wg_server_cmds)
for bad_path in ("etc/foo", "/etc/foo; rm -rf /", "/etc/$(reboot)", "/etc/a b", ""):
    try:
        write_file_cmd(bad_path, "x"); assert False, f"bad path must raise: {bad_path!r}"
    except ValueError: pass
for bad_owner in ("root", "root:root; reboot", "root root", ""):
    try:
        write_file_cmd("/etc/foo", "x", owner=bad_owner); assert False, f"bad owner must raise: {bad_owner!r}"
    except ValueError: pass
for bad_mode in ("777x", "9", "rwxr-xr-x", "0 644"):
    try:
        write_file_cmd("/etc/foo", "x", mode=bad_mode); assert False, f"bad mode must raise: {bad_mode!r}"
    except ValueError: pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()